In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

data = [
     (1, "C001", "Laptop", "50000", "2024-01-01"),
    (2, "C002", "Mobile", None, "2024-01-02"),
    (3, "C003", "Tablet", "20000", "2024-01-03"),
    (4, "C004", "Laptop", "55000", "2024-01-04"),
    (5, "C005", "Headphones", None, "2024-01-05"),
    (6, "C006", "Camera", "30000", "2024-01-06"),
    (7, "C007", "Mobile", "18000", "2024-01-07"),
    (8, "C008", "Watch", "8000", "2024-01-07")
]

columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

df = spark.createDataFrame(data, columns)

In [0]:
from pyspark.sql.functions import coalesce, col, lit

df = df.withColumn("amount", coalesce(col("amount"), lit(1000))).show()

+--------+-----------+----------+------+------------+
|order_id|customer_id|   product|amount|updated_date|
+--------+-----------+----------+------+------------+
|       1|       C001|    Laptop| 50000|  2024-01-01|
|       2|       C002|    Mobile|  1000|  2024-01-02|
|       3|       C003|    Tablet| 20000|  2024-01-03|
|       4|       C004|    Laptop| 55000|  2024-01-04|
|       5|       C005|Headphones|  1000|  2024-01-05|
|       6|       C006|    Camera| 30000|  2024-01-06|
|       7|       C007|    Mobile| 18000|  2024-01-07|
|       8|       C008|     Watch|  8000|  2024-01-07|
+--------+-----------+----------+------+------------+



In [0]:
from pyspark.sql.functions import to_date, col
df.withColumn("amount", col("amount").cast("int") ).withColumn("updated_date", to_date(col("updated_date"))).show()

+--------+-----------+----------+------+------------+
|order_id|customer_id|   product|amount|updated_date|
+--------+-----------+----------+------+------------+
|       1|       C001|    Laptop| 50000|  2024-01-01|
|       2|       C002|    Mobile|  NULL|  2024-01-02|
|       3|       C003|    Tablet| 20000|  2024-01-03|
|       4|       C004|    Laptop| 55000|  2024-01-04|
|       5|       C005|Headphones|  NULL|  2024-01-05|
|       6|       C006|    Camera| 30000|  2024-01-06|
|       7|       C007|    Mobile| 18000|  2024-01-07|
|       8|       C008|     Watch|  8000|  2024-01-07|
+--------+-----------+----------+------+------------+



Task 3: Add Derived Columns
 bonus = amount * 0.1
 category:
 20000 → High
 else → Low


In [0]:
df=df.withColumn("bonus",col("amount")* 0.1).show()

+--------+-----------+----------+------+------------+------+
|order_id|customer_id|   product|amount|updated_date| bonus|
+--------+-----------+----------+------+------------+------+
|       1|       C001|    Laptop| 50000|  2024-01-01|5000.0|
|       2|       C002|    Mobile|  NULL|  2024-01-02|  NULL|
|       3|       C003|    Tablet| 20000|  2024-01-03|2000.0|
|       4|       C004|    Laptop| 55000|  2024-01-04|5500.0|
|       5|       C005|Headphones|  NULL|  2024-01-05|  NULL|
|       6|       C006|    Camera| 30000|  2024-01-06|3000.0|
|       7|       C007|    Mobile| 18000|  2024-01-07|1800.0|
|       8|       C008|     Watch|  8000|  2024-01-07| 800.0|
+--------+-----------+----------+------+------------+------+



In [0]:
from pyspark.sql.functions import when, col

df = df.withColumn("category", when(col("amount") > 20000, "High").otherwise("Low")).show()

+--------+-----------+----------+------+------------+--------+
|order_id|customer_id|   product|amount|updated_date|category|
+--------+-----------+----------+------+------------+--------+
|       1|       C001|    Laptop| 50000|  2024-01-01|    High|
|       2|       C002|    Mobile|  NULL|  2024-01-02|     Low|
|       3|       C003|    Tablet| 20000|  2024-01-03|     Low|
|       4|       C004|    Laptop| 55000|  2024-01-04|    High|
|       5|       C005|Headphones|  NULL|  2024-01-05|     Low|
|       6|       C006|    Camera| 30000|  2024-01-06|    High|
|       7|       C007|    Mobile| 18000|  2024-01-07|     Low|
|       8|       C008|     Watch|  8000|  2024-01-07|     Low|
+--------+-----------+----------+------+------------+--------+



### Task 4: Apply UDF
 Create amount_bucket:
 < 10000 → Low
 10000–30000 → Medium
 30000 → High

In [0]:

from pyspark.sql.functions import udf,col,when
from pyspark.sql.types import StringType
df = df.withColumn(
    "amount",
    when(col("amount").isNull(), 1000).otherwise(col("amount"))
)
df = df.withColumn("amount", col("amount").cast("int"))
def bucket(x):
  if x< 10000:
    return "Low"
  elif x <=30000:
    return "Medium"
  else:
    return "High"

bucket_udf = udf(bucket, StringType())
df = df.withColumn("amount_bucket", bucket_udf(col("amount")))
df.show()

+--------+-----------+----------+------+------------+-------------+
|order_id|customer_id|   product|amount|updated_date|amount_bucket|
+--------+-----------+----------+------+------------+-------------+
|       1|       C001|    Laptop| 50000|  2024-01-01|         High|
|       2|       C002|    Mobile|  1000|  2024-01-02|          Low|
|       3|       C003|    Tablet| 20000|  2024-01-03|       Medium|
|       4|       C004|    Laptop| 55000|  2024-01-04|         High|
|       5|       C005|Headphones|  1000|  2024-01-05|          Low|
|       6|       C006|    Camera| 30000|  2024-01-06|       Medium|
|       7|       C007|    Mobile| 18000|  2024-01-07|       Medium|
|       8|       C008|     Watch|  8000|  2024-01-07|          Low|
+--------+-----------+----------+------+------------+-------------+



### Task 5: Full Load
 Load all data to target

In [0]:
df.write.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("default.target_table")

### Task 6: Incremental Load
 Load only new/updated records


In [0]:
last_loaded_date = "2024-01-01"

df_inc = df.filter(col("updated_date") > last_loaded_date).show()

+--------+-----------+----------+------+------------+-------------+
|order_id|customer_id|   product|amount|updated_date|amount_bucket|
+--------+-----------+----------+------+------------+-------------+
|       2|       C002|    Mobile|  1000|  2024-01-02|          Low|
|       3|       C003|    Tablet| 20000|  2024-01-03|       Medium|
|       4|       C004|    Laptop| 55000|  2024-01-04|         High|
|       5|       C005|Headphones|  1000|  2024-01-05|          Low|
|       6|       C006|    Camera| 30000|  2024-01-06|       Medium|
|       7|       C007|    Mobile| 18000|  2024-01-07|       Medium|
|       8|       C008|     Watch|  8000|  2024-01-07|          Low|
+--------+-----------+----------+------+------------+-------------+



Handle duplicates (keep latest)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("order_id").orderBy(col("updated_date").desc())
df_inc = df.filter(col("updated_date") > last_loaded_date)
df_dedup = df_inc.withColumn("rn", row_number().over(window)) \
                 .filter(col("rn") == 1) \
                 .drop("rn")

df_dedup.show()

+--------+-----------+----------+------+------------+-------------+
|order_id|customer_id|   product|amount|updated_date|amount_bucket|
+--------+-----------+----------+------+------------+-------------+
|       2|       C002|    Mobile|  1000|  2024-01-02|          Low|
|       3|       C003|    Tablet| 20000|  2024-01-03|       Medium|
|       4|       C004|    Laptop| 55000|  2024-01-04|         High|
|       5|       C005|Headphones|  1000|  2024-01-05|          Low|
|       6|       C006|    Camera| 30000|  2024-01-06|       Medium|
|       7|       C007|    Mobile| 18000|  2024-01-07|       Medium|
|       8|       C008|     Watch|  8000|  2024-01-07|          Low|
+--------+-----------+----------+------+------------+-------------+



In [0]:
df_dedup.write.format("delta") \
.mode("append") \
.saveAsTable("default.target_table")

### Task 7: Parameterization
 Pass:

input path

last_loaded_date

In [0]:

data = [
     (1, "C001", "Laptop", "50000", "2024-01-01"),
    (2, "C002", "Mobile", None, "2024-01-02"),
    (3, "C003", "Tablet", "20000", "2024-01-03"),
    (4, "C004", "Laptop", "55000", "2024-01-04"),
    (5, "C005", "Headphones", None, "2024-01-05"),
    (6, "C006", "Camera", "30000", "2024-01-06"),
    (7, "C007", "Mobile", "18000", "2024-01-07"),
    (8, "C008", "Watch", "8000", "2024-01-07")
]

columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

df = spark.createDataFrame(data, columns)

In [0]:
last_loaded_date = "2024-01-01"

df_inc = df.filter(col("updated_date") > last_loaded_date).show()

+--------+-----------+----------+------+------------+
|order_id|customer_id|   product|amount|updated_date|
+--------+-----------+----------+------+------------+
|       2|       C002|    Mobile|  NULL|  2024-01-02|
|       3|       C003|    Tablet| 20000|  2024-01-03|
|       4|       C004|    Laptop| 55000|  2024-01-04|
|       5|       C005|Headphones|  NULL|  2024-01-05|
|       6|       C006|    Camera| 30000|  2024-01-06|
|       7|       C007|    Mobile| 18000|  2024-01-07|
|       8|       C008|     Watch|  8000|  2024-01-07|
+--------+-----------+----------+------+------------+

